# HLT 2026/27 Group 2: Countdown GRPO - Direct Training Ablation (No Curriculum)

This notebook trains `Qwen/Qwen2.5-1.5B-Instruct` with GRPO **directly on 3-number Countdown puzzles for 200 steps** without the Stage 1 (2-number) curriculum bootstrapping phase.

### Purpose of this Ablation Study:
- **Scientific Goal:** Test whether multi-step mathematical reasoning emerges when training directly on complex problems (cold-start with sparse rewards), or if curriculum bootstrapping (Stage 1 on 2-number puzzles) is strictly necessary.
- **Comparison Target:** Compare accuracy and reward convergence against our Curriculum GRPO model (which achieved 97.4% on 2 numbers, 69.4% on 3 numbers, and 83.4% overall in-domain).

In [ ]:
# Cell 1: Environment Setup & Codebase Synchronization
# 1. Uninstall incompatible preinstalled torchao in Kaggle environment
!pip uninstall -y torchao
!pip install -q -U trl peft

# 2. Clone fresh repository and checkout working branch
%cd /kaggle/working
!rm -rf hlt-grpo-countdown
!git clone https://github.com/hamasli/hlt-grpo-countdown.git
%cd hlt-grpo-countdown
!git checkout fix/rewards-and-grpo-training

# 3. Verify CUDA availability and GPU acceleration
import torch, transformers, trl, peft
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
print(f"Versions: PyTorch {torch.__version__} | Transformers {transformers.__version__} | TRL {trl.__version__} | PEFT {peft.__version__}")

In [ ]:
# Cell 2: Dataset Generation and Verification Unit Tests
# Generates identical reproducible data splits: train.jsonl, val.jsonl, test.jsonl, test_ood.jsonl
!python -m src.data.generate

# Run unit test suite to verify reward functions, math parser, and format validation
!python -m pytest -q

In [ ]:
# Cell 3: Direct GRPO Training on 3-Number Puzzles (Ablation without Stage 1)
# Note: We do NOT provide --init_adapter, training fresh LoRA weights directly on levels 3
!python -m src.train.grpo_train \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --levels 3 \
    --max_steps 200 \
    --output_dir checkpoints/direct_3numbers

In [ ]:
# Cell 4: Evaluation of Direct GRPO Model on Test Sets
# 1. In-Domain Test Set Evaluation (2 and 3 numbers)
print("=== 1. In-Domain Evaluation (data/test.jsonl: 2 and 3 numbers) ===")
!python -m src.eval.evaluate \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --method grpo \
    --adapter checkpoints/direct_3numbers/final \
    --data data/test.jsonl

# 2. Out-Of-Domain Test Set Evaluation (4 numbers - Generalization)
print("\n=== 2. Out-of-Domain Evaluation (data/test_ood.jsonl: 4 numbers) ===")
!python -m src.eval.evaluate \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --method grpo \
    --adapter checkpoints/direct_3numbers/final \
    --data data/test_ood.jsonl

In [ ]:
# Cell 5: Archive Logs and Evaluation Outputs for Download
!tar -czvf direct_experiment_results.tar.gz results/*direct* checkpoints/direct_3numbers/*.json
print("\nDirect training logs archived into direct_experiment_results.tar.gz")